In [3]:
import pandas as pd
import numpy as nd
import matplotlib as ply
import seaborn as sns


In [4]:
 hr1= pd.read_csv(r"C:\Users\Anugraha_Jay\OneDrive\Documents\Anugraha_Jayakumar\PROJECTS\PYTHON\Hr Atrrition\hr_train.csv")

In [5]:
hr1.describe()

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years
count,10499.000000,10499.000000,10499.000000,10499.000000,10499.000000,10499.000000,10499.000000,10499.000000
mean,0.612683,0.717131,3.808553,201.059815,3.494238,0.144299,0.292885,0.021716
std,0.248578,0.171483,1.230572,49.959332,1.453227,0.351410,0.455108,0.145763
min,0.090000,0.360000,2.000000,96.000000,2.000000,0.000000,0.000000,0.000000
25%,0.440000,0.560000,3.000000,156.000000,3.000000,0.000000,0.000000,0.000000
50%,0.640000,0.720000,4.000000,200.000000,3.000000,0.000000,0.000000,0.000000
75%,0.820000,0.870000,5.000000,245.000000,4.000000,0.000000,1.000000,0.000000
max,1.000000,1.000000,7.000000,310.000000,10.000000,1.000000,1.000000,1.000000


# EDA

### 1) What share of the workforce qualifies as a 'good employee' under a defined performance standard, and what does that group look like?

In [6]:
## good_employee = 1 if last_evaluation > mean AND number_project >= 4, else 0.

mean_last_eval= hr1['last_evaluation'].mean()
condition1 = hr1['last_evaluation'] > mean_last_eval
condition1.head()
hr1['good_employee']= ((hr1['last_evaluation'] > mean_last_eval)&(hr1['number_project'] >= 4)).astype(int)
print(hr1['good_employee'].value_counts())




good_employee
0    6771
1    3729
Name: count, dtype: int64


In [7]:
## what % of the total workforce
good_employee= hr1['good_employee'].value_counts(normalize=True)*100
print(good_employee)


good_employee
0    64.485714
1    35.514286
Name: proportion, dtype: float64


### 2) Do "good employees" leave (attrition) at a different rate than everyone else?

In [8]:
 hr1.groupby('good_employee')['left'].mean()*100

good_employee
0    24.431315
1    38.106731
Name: left, dtype: float64

In [9]:
left_rate=hr1.groupby('good_employee')['left'].mean()*100
diff = left_rate.loc[1] - left_rate.loc[0]
print(diff)

13.675416403746755


### 3) Is there a tenure "danger zone" for attrition among good employees?

In [10]:
tenure= hr1.groupby(['good_employee','time_spend_company'])['left'].mean()*100
print(tenure)


good_employee  time_spend_company
0              2.0                   10.661765
               3.0                   34.870149
               4.0                   17.245509
               5.0                   22.418879
               6.0                   15.116279
               7.0                   10.204082
               8.0                    8.536585
               10.0                   9.090909
1              2.0                   12.328767
               3.0                   10.984848
               4.0                   59.875905
               5.0                   71.676301
               6.0                   55.465587
               7.0                   11.111111
               8.0                   10.344828
               10.0                  11.111111
Name: left, dtype: float64


### 4) Among good employees, do those who left show signs of overwork compared to those who stayed?

In [11]:
filter_good= hr1[hr1['good_employee']==1]
print(filter_good)


       satisfaction_level  last_evaluation  number_project  \
0                    0.09             0.92             7.0   
1                    0.09             0.80             7.0   
3                    0.09             0.89             6.0   
4                    0.09             0.94             6.0   
5                    0.09             0.91             6.0   
...                   ...              ...             ...   
10484                1.00             0.77             5.0   
10485                1.00             0.87             4.0   
10491                1.00             0.99             4.0   
10494                1.00             0.94             4.0   
10495                1.00             0.97             5.0   

       average_montly_hours  time_spend_company  Work_accident  left  \
0                     301.0                 4.0            0.0   1.0   
1                     283.0                 5.0            0.0   1.0   
3                     282.0            

In [12]:
overwork = filter_good.groupby('left')['average_montly_hours'].mean()
print(overwork)

left
0.0    206.506066
1.0    248.411682
Name: average_montly_hours, dtype: float64


In [13]:
diff = overwork.loc[1] - overwork.loc[0]
print(diff)

41.90561605625936


### 5) Among good employees, do those who left report lower satisfaction than those who stayed?

In [14]:
satisfaction = filter_good.groupby('left')['satisfaction_level'].mean()
print(satisfaction)

left
0.0    0.657426
1.0    0.489184
Name: satisfaction_level, dtype: float64


In [15]:
diff1= satisfaction.loc[1] - satisfaction.loc[0]
print(diff1)

-0.16824266968485835


## EDA FINDINGS



### Defining a "Good Employee"
A good employee is defined as one whose `last_evaluation` is above the dataset mean 
**and** who has completed **4 or more projects** (`number_project >= 4`). 
Note: `left` was deliberately excluded from this definition to avoid circular reasoning 
when studying attrition patterns among good employees.

---

### EDA Question 1: What share of the workforce qualifies as a "good employee"?
**~35.5% of employees (3,729 out of 10,499)** meet the "good employee" criteria.

---

### EDA Question 2: Do good employees leave at a different rate than everyone else?
- Good employees (`good_employee = 1`): **38.1%** left
- Regular employees (`good_employee = 0`): **24.4%** left
- **Gap: ~13.7 percentage points** — good employees leave at a notably higher rate.

---

### EDA Question 3: Is there a tenure "danger zone" for attrition among good employees?
Among good employees, attrition is heavily concentrated in **years 4–6 of tenure**, 
**peaking at year 5 (~71.7%)**. This is a much sharper, more concentrated spike than 
regular employees show, whose attrition is comparatively mild and peaks earlier 
(year 3, ~34.9%) without a dramatic surge.

---

### EDA Question 4: Among good employees, do those who left show signs of overwork?
- Good employees who left: averaged **~248 hours/month**
- Good employees who stayed: averaged **~207 hours/month**
- **Gap: ~42 hours/month (~10 extra hours/week)** — good leavers show a strong 
  overwork signal compared to those who stayed, suggesting burnout as a contributing factor.

---

### EDA Question 5: Among good employees, do those who left report lower satisfaction than those who stayed?
- Good employees who left: satisfaction averaged **0.49**
- Good employees who stayed: satisfaction averaged **0.66**
- **Gap: ~0.17** — good leavers reported meaningfully lower satisfaction, though not 
  extremely low in absolute terms (still mid-scale, not close to 0).

---

### Overall Narrative
Good employees leave at a significantly higher rate than the rest of the workforce, 
with attrition sharply concentrated around years 4–6 of tenure. Those who leave show 
both a strong overwork signal (higher average monthly hours) and moderately lower 
satisfaction compared to good employees who stay — together pointing toward burnout 
as a plausible, though not definitively proven, driver of attrition among top performers.

*(Note: these are correlational findings from EDA, not causal claims.)*

## Forcasting

In [88]:
hr0=pd.read_csv(r"C:\Users\Anugraha_Jay\OneDrive\Documents\Anugraha_Jayakumar\PROJECTS\PYTHON\Hr Atrrition\hr_train.csv")

#### Cleaning 

In [89]:
hr0[hr0['sales']=="sales"]


,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,sales,salary
5,0.09,0.91,6.0,248.0,4.0,0.0,1.0,0.0,sales,low
7,0.09,0.93,7.0,270.0,4.0,0.0,1.0,0.0,sales,medium
8,0.09,0.79,6.0,293.0,5.0,0.0,1.0,0.0,sales,low
13,0.09,0.95,6.0,292.0,4.0,0.0,1.0,0.0,sales,medium
15,0.09,0.79,6.0,275.0,4.0,0.0,0.0,0.0,sales,low
...,...,...,...,...,...,...,...,...,...,...
10483,1.00,0.39,2.0,210.0,5.0,0.0,0.0,0.0,sales,low
10484,1.00,0.77,5.0,269.0,3.0,0.0,0.0,0.0,sales,low
10487,1.00,0.61,6.0,270.0,3.0,0.0,0.0,0.0,sales,low
10488,1.00,0.49,4.0,140.0,3.0,0.0,0.0,0.0,sales,low


In [90]:
# changeing column name "sales to"dapertment" as there is a sales dept within this colum and it get lost while concating
hr0 = hr0.rename(columns={"sales": "department"})
hr0.head(2)

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,department,salary
0,0.09,0.92,7.0,301.0,4.0,0.0,1.0,0.0,marketing,low
1,0.09,0.80,7.0,283.0,5.0,0.0,1.0,0.0,technical,low


In [91]:
# cleaning data of nans
hr0= hr0.drop(10499, axis =0)

In [92]:
### create dummines
salary_dummy= pd.get_dummies(hr0['salary'],drop_first=True)
dept_dummy= pd.get_dummies(hr0['department'],drop_first= True)
hr2= pd.concat([hr0,salary_dummy,sales_dummy],axis=1)
hr2.head(3)

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,department,salary,...,medium,RandD,accounting,hr,management,marketing,product_mng,sales,support,technical
0,0.09,0.92,7.0,301.0,4.0,0.0,1.0,0.0,marketing,low,...,False,False,False,False,False,True,False,False,False,False
1,0.09,0.80,7.0,283.0,5.0,0.0,1.0,0.0,technical,low,...,False,False,False,False,False,False,False,False,False,True
2,0.09,0.62,6.0,294.0,4.0,0.0,1.0,0.0,accounting,low,...,False,False,True,False,False,False,False,False,False,False


In [93]:
hr3= hr2.drop(['department','salary'], axis=1)
hr3.head(3)

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,low,medium,RandD,accounting,hr,management,marketing,product_mng,sales,support,technical
0,0.09,0.92,7.0,301.0,4.0,0.0,1.0,0.0,True,False,False,False,False,False,True,False,False,False,False
1,0.09,0.80,7.0,283.0,5.0,0.0,1.0,0.0,True,False,False,False,False,False,False,False,False,False,True
2,0.09,0.62,6.0,294.0,4.0,0.0,1.0,0.0,True,False,False,True,False,False,False,False,False,False,False


In [95]:
## train-test split
from sklearn.model_selection import train_test_split
x= hr3.drop('left', axis=1)
y=hr3['left']
x_train, x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=121)

In [96]:
x_train.isna().sum()


satisfaction_level       0
last_evaluation          0
number_project           0
average_montly_hours     0
time_spend_company       0
Work_accident            0
promotion_last_5years    0
low                      0
medium                   0
RandD                    0
accounting               0
hr                       0
management               0
marketing                0
product_mng              0
sales                    0
support                  0
technical                0
dtype: int64

In [97]:
## Logistic Regression Model
from sklearn.linear_model import LogisticRegression
model=LogisticRegression()

## Scaling puts all features on a similar numeric range so the model can converge properly, 
#since raw columns like hours(up to 310) vs satisfaction (0-1) are on very different scales

from sklearn.preprocessing import StandardScaler
scaler= StandardScaler()
x_train_scaled= scaler.fit_transform(x_train)
x_test_scaled= scaler.transform(x_test)

hr_log_reg =model.fit(x_train_scaled,y_train)

## Prediction on test data
y_pred=model.predict(x_test_scaled)

In [98]:
#validation

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score
accuracy= accuracy_score(y_test,y_pred)
print(accuracy)

cm= confusion_matrix(y_test,y_pred)
print(cm)

report = classification_report(y_test, y_pred)
print(report)

0.7161904761904762
[[1365  114]
 [ 482  139]]
              precision    recall  f1-score   support

         0.0       0.74      0.92      0.82      1479
         1.0       0.55      0.22      0.32       621

    accuracy                           0.72      2100
   macro avg       0.64      0.57      0.57      2100
weighted avg       0.68      0.72      0.67      2100



In [99]:
## AUC cumputation
## We use predict_proba instead of predict because AUC needs 
#the probability of leaving (e.g. 73%), not just a final yes/no answer.

y_prob=model.predict_proba(x_test_scaled)
y_prob[:5]

array([[0.68720711, 0.31279289],
       [0.75903093, 0.24096907],
       [0.72534636, 0.27465364],
       [0.69273152, 0.30726848],
       [0.92662487, 0.07337513]])

In [100]:
# you need only the predicted probabilities for whose who left
y_prob_left=y_prob[:,1]
auc=roc_auc_score(y_test,y_prob_left)
print(auc)

0.7277673799265945


### Part 2: Model Building — Logistic Regression (Baseline)

**Interpretation:** Accuracy looks decent on the surface, but the model only catches 
**22% of employees who actually leave** — it misses the vast majority of real leavers. 
AUC of 0.73 shows moderate but not strong ability to rank employees by risk. This confirms 
logistic regression is likely insufficient as a final model for the actual HR goal of 
identifying at-risk employees.

## Decision Tree

In [101]:
from sklearn.tree import DecisionTreeClassifier
model2 = DecisionTreeClassifier()


In [102]:
#model
model2.fit(x_train,y_train)
## Prediction on test data
y_pred2=model2.predict(x_test)
print(y_pred2)

[0. 0. 0. ... 1. 0. 0.]


In [103]:
#validation
accuracy2= accuracy_score(y_test,y_pred2)
print(accuracy2)

cm2= confusion_matrix(y_test,y_pred2)
print(cm2)

report2 = classification_report(y_test, y_pred2)
print(report2)

0.7842857142857143
[[1238  241]
 [ 212  409]]
              precision    recall  f1-score   support

         0.0       0.85      0.84      0.85      1479
         1.0       0.63      0.66      0.64       621

    accuracy                           0.78      2100
   macro avg       0.74      0.75      0.74      2100
weighted avg       0.79      0.78      0.79      2100



In [104]:
# AUC
y_prob2=model2.predict_proba(x_test)[:,1]
auc2=roc_auc_score(y_test,y_prob2)
print(auc2)

0.7577708966867329


In [105]:
train_pred2 = model2.predict(x_train)
train_accuracy2 = accuracy_score(y_train, train_pred2)
print(train_accuracy2)

0.9795213715918561


#### There is a 97.9% accuracy on the train data. This means that, when it is tested on its train data itself, the model showed almost 98% accuracy, but when it comes to the test data, it has fallen down to 76%. That is exactly a signal towards overfitting, and hence we are going to random forest.

## Random Forest


In [106]:
from sklearn.ensemble import RandomForestClassifier
model3= RandomForestClassifier()

## model
model3.fit(x_train,y_train)

## Prediction on test data
y_pred3=model3.predict(x_test)

In [107]:
#validation
accuracy3= accuracy_score(y_test,y_pred3)
print(accuracy3)

cm3=confusion_matrix(y_test,y_pred3)
print(cm3)

report3 = classification_report(y_test, y_pred3)
print(report3)

# AUC
y_prob3=model3.predict_proba(x_test)[:,1]
auc3=roc_auc_score(y_test,y_prob3)
print(auc3)

0.86
[[1397   82]
 [ 212  409]]
              precision    recall  f1-score   support

         0.0       0.87      0.94      0.90      1479
         1.0       0.83      0.66      0.74       621

    accuracy                           0.86      2100
   macro avg       0.85      0.80      0.82      2100
weighted avg       0.86      0.86      0.85      2100

0.8344683867216718


In [108]:
train_pred3 = model3.predict(x_train)
train_accuracy3 = accuracy_score(y_train, train_pred3)
print(train_accuracy3)

0.9794023097987856


## Comparison

In [109]:
from sklearn.metrics import precision_recall_fscore_support

In [110]:
precision, recall, f1, support = precision_recall_fscore_support(y_test, y_pred, average=None)
precision2, recall2, f12, support2 = precision_recall_fscore_support(y_test, y_pred2, average=None)
precision3, recall3, f13, support3 = precision_recall_fscore_support(y_test, y_pred3, average=None)

comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision (left)', 'Recall (left)', 'AUC'],
    'Logistic Regression': [accuracy, precision[1], recall[1], auc],
    'Decision Tree': [accuracy2, precision2[1], recall2[1], auc2],
    'Random Forest': [accuracy3, precision3[1], recall3[1], auc3]
})
print(comparison)

             Metric  Logistic Regression  Decision Tree  Random Forest
0          Accuracy             0.716190       0.784286       0.860000
1  Precision (left)             0.549407       0.629231       0.832994
2     Recall (left)             0.223833       0.658615       0.658615
3               AUC             0.727767       0.757771       0.834468


### Forcast-Summary

Random forest was the clear winner among the three models, improving on accuracy, 
precision, and AUC by a wide margin over both logistic regression and the standalone 
decision tree. Recall for employees who left was the one metric that did not improve 
further — it matched the decision tree's 0.66 rather than exceeding it, meaning even 
the best model still misses roughly a third of actual leavers. Overall, random forest 
is the strongest and most reliable model of the three, but not a fully solved prediction 
problem — there's still real room for improvement in catching every at-risk employee.

## LLM integration

## Part A: Individual Employee Predictor
Given one employee's attributes, predicts their attrition probability and uses SHAP + an LLM to explain the "why" in plain language for HR; supports a single appraisal-style conversation.


In [67]:
pd.DataFrame({"satisfaction_level":[0.4],"last_evaluation":[0.8]})

,satisfaction_level,last_evaluation
0,0.4,0.8


In [111]:
print(salary_dummy.columns.tolist())
print(dept_dummy.columns.tolist())

['low', 'medium']
['RandD', 'accounting', 'hr', 'management', 'marketing', 'product_mng', 'sales', 'support', 'technical']


In [113]:
print(model3.feature_names_in_)

['satisfaction_level' 'last_evaluation' 'number_project'
 'average_montly_hours' 'time_spend_company' 'Work_accident'
 'promotion_last_5years' 'low' 'medium' 'RandD' 'accounting' 'hr'
 'management' 'marketing' 'product_mng' 'sales' 'support' 'technical']


In [115]:
employee_status=pd.DataFrame({"satisfaction_level":[0.4],"last_evaluation":[0.8],"number_project":[5], "average_montly_hours":[200], "time_spend_company":[5], "Work_accident":[0], "promotion_last_5years":[1],"low":[1], "medium":[0],"RandD":[0], "accounting":[0],"hr":[0], "management":[1], "marketing":[0], "product_mng":[0], "sales":[0], "support":[0], "technical":[0]})
print(employee_status)

   satisfaction_level  last_evaluation  number_project  average_montly_hours  \
0                 0.4              0.8               5                   200   

   time_spend_company  Work_accident  promotion_last_5years  low  medium  \
0                   5              0                      1    1       0   

   RandD  accounting  hr  management  marketing  product_mng  sales  support  \
0      0           0   0           1          0            0      0        0   

   technical  
0          0  


In [116]:
model3.predict(employee_status)

array([0.])

##### means the model predicted class 0 for this employee — meaning it predicts they will not leave

In [144]:
model3.predict_proba(employee_status)


array([[0.8885, 0.1115]])

In [147]:
prob_leaving=model3.predict_proba(employee_status)[:,1]
print(prob_leaving)

[0.1115]


In [122]:
!pip install shap
import shap
explainer= shap.TreeExplainer(model3)


   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   ---------------------------------------- 2/2 [shap]



### SHAP explains why the model made a specific prediction for one individual, by showing how much each feature value pushed that prediction up or down from the average — unlike feature importance, which only shows what matters across the whole dataset, not for one person.

In [124]:
shap_values = explainer.shap_values(employee_status)
print(shap_values)

[[[ 0.0346225  -0.0346225 ]
  [ 0.07077342 -0.07077342]
  [ 0.04231819 -0.04231819]
  [ 0.08972248 -0.08972248]
  [-0.03446543  0.03446543]
  [ 0.00084802 -0.00084802]
  [-0.00812437  0.00812437]
  [-0.02266701  0.02266701]
  [ 0.00229046 -0.00229046]
  [ 0.0024991  -0.0024991 ]
  [ 0.00070738 -0.00070738]
  [ 0.00053897 -0.00053897]
  [-0.01425081  0.01425081]
  [ 0.00190488 -0.00190488]
  [ 0.00093895 -0.00093895]
  [ 0.00741028 -0.00741028]
  [ 0.00285527 -0.00285527]
  [ 0.00329592 -0.00329592]]]


In [125]:
leave_values = shap_values[0, :, 1]
print(leave_values)
## 0 — picks employee number 0 (your only employee, since you only have one row). This says "give me the first, and only, employee."
## : — the colon means "give me all of them," so this says "give me all 17 features, don't skip any."
## 1 — picks index 1 within the class pair. Remember each feature had two numbers, [stay_value, leave_value] at positions 0 and 1. So 1 says "give me only the leave value, not the stay value."

[-0.0346225  -0.07077342 -0.04231819 -0.08972248  0.03446543 -0.00084802
  0.00812437  0.02266701 -0.00229046 -0.0024991  -0.00070738 -0.00053897
  0.01425081 -0.00190488 -0.00093895 -0.00741028 -0.00285527 -0.00329592]


In [128]:
model3.feature_names_in_

array(['satisfaction_level', 'last_evaluation', 'number_project',
       'average_montly_hours', 'time_spend_company', 'Work_accident',
       'promotion_last_5years', 'low', 'medium', 'RandD', 'accounting',
       'hr', 'management', 'marketing', 'product_mng', 'sales', 'support',
       'technical'], dtype=object)

In [134]:
#creating dataframe for features and its shap values
attrition=pd.DataFrame({"Features":model3.feature_names_in_,"shap_values":leave_values})
attrition["shap_abs"]= attrition["shap_values"].abs()
attrition=attrition.sort_values("shap_abs",ascending=False)
print(attrition)

                 Features  shap_values  shap_abs
3    average_montly_hours    -0.089722  0.089722
1         last_evaluation    -0.070773  0.070773
2          number_project    -0.042318  0.042318
0      satisfaction_level    -0.034623  0.034623
4      time_spend_company     0.034465  0.034465
7                     low     0.022667  0.022667
12             management     0.014251  0.014251
6   promotion_last_5years     0.008124  0.008124
15                  sales    -0.007410  0.007410
17              technical    -0.003296  0.003296
16                support    -0.002855  0.002855
9                   RandD    -0.002499  0.002499
8                  medium    -0.002290  0.002290
13              marketing    -0.001905  0.001905
14            product_mng    -0.000939  0.000939
5           Work_accident    -0.000848  0.000848
10             accounting    -0.000707  0.000707
11                     hr    -0.000539  0.000539


In [135]:
top_features = attrition.head(5)
print(top_features)

               Features  shap_values  shap_abs
3  average_montly_hours    -0.089722  0.089722
1       last_evaluation    -0.070773  0.070773
2        number_project    -0.042318  0.042318
0    satisfaction_level    -0.034623  0.034623
4    time_spend_company     0.034465  0.034465


In [138]:
x=5
print(f"The value is {x}")

The value is 5


In [150]:
!pip install groq


   -------------------- ------------------- 1/2 [groq]
   ---------------------------------------- 2/2 [groq]



In [157]:
prompt= f"Here is an employee's attrition probability and the top features driving it. Write a short, plain-language explanation for an HR audience.\n\nProbability of leaving: {prob_leaving}\n\nTop contributing factors:\n{top_features}"

In [298]:

import os
os.path.exists(".env")
!pip install python-dotenv
with open(".gitignore", "w") as f:
    f.write(".env")

from groq import Groq
import os
from dotenv import load_dotenv

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))
os.path.exists(".env")

True

In [299]:
from groq import Groq
import os
client= Groq(api_key=os.getenv("GROQ_API_KEY"))
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role":"user","content":prompt}
    ]
)
from IPython.display import Markdown, display

display(Markdown(response.choices[0].message.content))

**HR Summary – Employees with last_evaluation < 0.5**

---

### 1. Overall risk level
- The segment contains **1,080 employees** and the model predicts an **average attrition probability of about 42 %**.  
- In plain terms, roughly **four out of every ten** employees in this group are likely to leave within the next period if nothing changes.

### 2. How risk is distributed
- **High‑risk employees (>70% chance):** 373 (≈ 35 % of the segment)  
- **Medium‑risk employees (30‑70% chance):** 108 (≈ 10 %)  
- **Low‑risk employees (<30% chance):** 599 (≈ 55 %)

**Interpretation:** Risk is **concentrated** in a sizeable minority—over a third of the group falls into the high‑risk category, while the remainder is split between low and medium risk. The distribution is not even; a substantial core of employees is at immediate danger of leaving.

### 3. Potential business impact if left unchecked
- With an average 42 % attrition likelihood, **nearly half of these 1,080 staff members could exit**.  
- This would mean a loss of **institutional knowledge and productivity** from the high‑risk cohort (373 employees) plus additional turnover from the medium‑risk group.  
- The organization would face the **direct costs of recruiting and onboarding** for each departure, plus the **indirect cost of disrupted workflows** and reduced team performance.

### 4. Practical retention actions for this segment
- **Targeted development plans**  
  - Pair high‑risk employees with mentors or coaches to address performance gaps reflected by low evaluations.  
  - Set short‑term, measurable goals and provide frequent feedback to rebuild confidence and engagement.

- **Recognition and incentive review**  
  - Conduct quick pulse surveys to identify specific concerns (e.g., lack of recognition, unclear career path).  
  - Offer tailored short‑term incentives (e.g., spot bonuses, skill‑building stipends) that align with the employee’s role and performance level.

- **Improved onboarding/skill‑gap support**  
  - For those with low evaluations, provide focused training modules or job‑shadowing opportunities to boost competency quickly.  
  - Track progress and celebrate improvements to demonstrate investment in their growth.

These steps focus on the **high‑risk core** while also supporting the broader group, helping to lower the overall attrition probability and protect the organization’s talent base.

### Employee Input Function
Collects a new employee's real-world stats via `input()`, converts categorical fields (salary, department) into the model's dummy-encoded format, and builds a one-row DataFrame ready for prediction — so the tool works for any employee, not just the one hardcoded example.

In [177]:
print(hr0['satisfaction_level'].min(), hr0['satisfaction_level'].max())
print(hr0["last_evaluation"].min(),hr0["last_evaluation"].max())
print(hr0["number_project"].min(),hr0["number_project"].max())
print(hr0["time_spend_company"].min(),hr0["time_spend_company"].max())
print(hr0["average_montly_hours"].min(),hr0["average_montly_hours"].max())

0.09 1.0
0.36 1.0
2.0 7.0
2.0 10.0
96.0 310.0


In [300]:
satisfaction_input=float(input("Enter satisfaction_level(0.09-1.0):"))
last_eval_input=float(input("Enter last_evaluation_score(0.00-1.0):"))
project_input=int(input("Enter num_projects (1-10):"))
time_input=int(input("Enter time_spend_company (1-10):"))
Work_accident_input=int(input("Enter Work_accident (0/1):"))
promotion_last_5years_input=int(input("Enter promotion_last_5years (0/1):"))
average_montly_hours_input=int(input("Enter average_montly_hours (90-330):"))
dept_input = input("Enter department (RandD/accounting/hr/IT/management/marketing/product_mng/sales/support/technical): ")

if dept_input== 'RandD':
    accounting= 0
    hr= 0
    marketing= 0
    product_mng=0
    sales=0
    management=0
    support=0
    technical=0
    RandD=1
elif dept_input== 'accounting':
    management=0
    accounting= 1
    hr= 0
    marketing= 0
    product_mng=0
    sales=0
    support=0
    technical=0
    RandD=0
elif dept_input== 'hr':
    management=0
    accounting= 0
    hr= 1
    marketing= 0
    product_mng=0
    sales=0
    support=0
    technical=0
    RandD=0
elif dept_input== 'marketing':
    management=0
    accounting= 0
    hr= 0
    marketing= 1
    product_mng=0
    sales=0
    support=0
    technical=0
    RandD=0

elif dept_input== 'product_mng':
    management=0
    accounting= 0
    hr= 0
    marketing= 0
    product_mng=1
    sales=0
    support=0
    technical=0
    RandD=0
elif dept_input== 'sales':
    management=0
    accounting= 0
    hr= 0
    marketing= 0
    product_mng=0
    sales=1
    support=0
    technical=0
    RandD=0
elif dept_input== 'IT':
    management=0
    accounting= 0
    hr= 0
    marketing= 0
    product_mng=0
    sales=0
    support=0
    technical=0
    RandD=0
elif dept_input== 'support':
    management=0
    accounting= 0
    hr= 0
    marketing= 0
    product_mng=0
    sales=0
    support=1
    technical=0
    RandD=0
elif dept_input== 'technical':
    management=0
    accounting= 0
    hr= 0
    marketing= 0
    product_mng=0
    sales=0
    support=0
    technical=1
    RandD=0    
elif dept_input== 'management':
    management=1
    accounting= 0
    hr= 0
    marketing= 0
    product_mng=0
    sales=0
    support=0
    technical=0
    RandD=0    
    


salary_input= input("Enter salary level(low/medium/high):")
if salary_input== "high":
    low=0
    medium=0
elif salary_input=="low":
   low=1
   medium = 0   
elif salary_input == "medium":
    low=0
    medium=1                    
    


employee_status=pd.DataFrame({"satisfaction_level":[satisfaction_input],"last_evaluation":[last_eval_input],"number_project":[project_input], "average_montly_hours":[average_montly_hours_input], "time_spend_company":[time_input], "Work_accident":[Work_accident_input], "promotion_last_5years":[promotion_last_5years_input],"low":[low], "medium":[medium],"RandD":[RandD], "accounting":[accounting],"hr":[hr], "management":[management], "marketing":[marketing], "product_mng":[product_mng], "sales":[sales], "support":[support], "technical":[technical]})
#print(employee_status)

model_ai=model3.predict_proba(employee_status)[:,1]
model3.feature_names_in_

# SHAP to explain which features contributed to the attrition probability
shap_values=explainer.shap_values(employee_status)[0,:,1]
#print(shap_values)

#matching featues to prob of leaving and creating a df for the same

attrition= pd.DataFrame({"features":model3.feature_names_in_,"shap_values":shap_values})
attrition["shap_values"]= attrition["shap_values"].abs()
attritiion=attrition.sort_values("shap_values", ascending= False)
#print(attritiion)
top_feautures= attrition.head(5)
#print(top_feautures)


prompt= f"Here is an employee's attrition probability and the top features driving it. Write a short, plain-language explanation for an HR audience.\n\nProbability of leaving: {model_ai}\n\nTop contributing factors:\n{top_features}\n\nEmployee Attributes:\n{employee_status}\n\nUse the employee's actual attribute values above (not just the direction of influence) to make your explanation specific and accurate to this employee — for example, state their actual tenure, hours, etc. where relevant."

from groq import Groq
client=Groq(api_key=os.getenv("GROQ_API_KEY"))
reponse= client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role":"user","content":prompt}]
)
from IPython.display import Markdown, display

display(Markdown(reponse.choices[0].message.content))



    



Enter satisfaction_level(0.09-1.0): 0.8
Enter last_evaluation_score(0.00-1.0): 0.7
Enter num_projects (1-10): 6
Enter time_spend_company (1-10): 240
Enter Work_accident (0/1): 0
Enter promotion_last_5years (0/1): 1
Enter average_montly_hours (90-330): 30
Enter department (RandD/accounting/hr/IT/management/marketing/product_mng/sales/support/technical):  it
Enter salary level(low/medium/high): high


**Attrition risk snapshot**

- **Overall chance of leaving:** about **20 %** (roughly 1 in 5).  
- **Why the model gives this number:** it looks at the employee’s specific data and weighs each factor that most influences the prediction.

**What the model is saying about this employee**

| Factor | What the employee’s record shows | How it pushes the risk up or down | Why it matters |
|--------|----------------------------------|-----------------------------------|----------------|
| **Time spent at the company** | **240 months** (≈ 20 years) | **Raises the risk** (positive SHAP value) | Very long tenure can lead to “career plateau” feelings or a desire for a new challenge. |
| **Average monthly work hours** | **30 hours** per month | **Lowers the risk** (negative SHAP value) | Working relatively few hours suggests a manageable workload, which tends to keep people around. |
| **Satisfaction level** | **0.8** (on a 0‑1 scale) | **Lowers the risk** | High job satisfaction is a strong retention driver. |
| **Last performance evaluation** | **0.7** (on a 0‑1 scale) | **Lowers the risk** | A solid recent review signals the employee is doing well and feels valued. |
| **Number of projects handled** | **6** projects | **Lowers the risk** | Being kept busy with a variety of projects can boost engagement. |
| **Promotion in the last 5 years** | **Yes (1)** | (not in top‑5 SHAP list, but a positive signal) | Recent advancement usually reduces turnover risk. |
| **Work accident, department, etc.** | No accidents; works in **product management** | (neutral to slightly positive) | No safety concerns; department fit is neutral for this prediction. |

**Take‑away for HR**

- The **single biggest driver of the 20 % attrition risk** is the employee’s **very long tenure** (20 years). Even though they are satisfied, evaluated well, and not overloaded, the length of service alone nudges the model toward a higher turnover probability.
- All other observed factors—low work hours, high satisfaction, solid performance, and a decent project load—**pull the risk down**, which is why the overall probability stays modest rather than high.

**Suggested actions**

1. **Career‑growth conversation:** Even though the employee was promoted recently, a 20‑year stay can create a feeling of “having peaked.” Discuss future career paths, stretch assignments, or leadership opportunities that align with their product‑management expertise.
2. **Recognition of tenure:** Acknowledge the milestone (e.g., a service award or special project) to reinforce that the company values their long‑term contribution.
3. **Check workload balance:** While 30 hours/month is low, confirm that the employee feels the work is meaningful and not under‑challenged.
4. **Stay interview:** Use a brief stay interview to surface any latent concerns (e.g., desire for a new industry, personal goals) that may not be captured by the current data.

By addressing the tenure‑related sentiment and continuing to nurture the positive factors already present, you can likely keep this employee’s attrition risk below the current 20 % level.

## Part B: Segment Risk Query
Given a filter criteria (e.g. low satisfaction, 5+ years tenure, etc), computes attrition risk across the matching group of employees ,average probability plus a high/medium/low risk breakdown; for broader retention strategy or planning decisions.

In [ ]:
hr3.head(1)

In [231]:
hr4=hr3.drop("left" ,axis=1)
hr4.head(1)

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,promotion_last_5years,low,medium,RandD,accounting,hr,management,marketing,product_mng,sales,support,technical
0,0.09,0.92,7.0,301.0,4.0,0.0,0.0,True,False,False,False,False,False,True,False,False,False,False


In [232]:
low_satisfaction_employees= hr4[hr4['satisfaction_level']<0.4]
print(len(low_satisfaction_employees))





2015


In [246]:
low_satisfaction_prob= model3.predict_proba(low_satisfaction_employees)[:,1]
print(len(low_satisfaction_prob))

2015


In [247]:
low_satisfaction_prob.mean()

np.float64(0.5455350171334042)

In [248]:
# creating Categories within this array
high_risk= low_satisfaction_prob[low_satisfaction_prob>0.7]
low_risk= low_satisfaction_prob[low_satisfaction_prob<0.3]
medium_risk= low_satisfaction_prob[(low_satisfaction_prob>=0.3) & (low_satisfaction_prob <= 0.7)]
print(len(high_risk),len(low_risk),len(medium_risk))
print(len(low_satisfaction_prob))

965 844 206
2015


In [265]:
prompt = f"""You are analyzing a segment of employees for an HR audience. Here is the data:

Filter applied: employees with satisfaction level below 0.4
Total employees in this group: {len(low_satisfaction_employees)}
Average predicted attrition probability for this group: {low_satisfaction_prob.mean()*100}%

Risk breakdown:
- High risk (>70% probability of leaving): {len(high_risk)} employees
- Medium risk (30-70% probability of leaving): {len(medium_risk)} employees
- Low risk (<30% probability of leaving): {len(low_risk)} employees

Using only the numbers above, write a plain-language summary for HR covering:
1. The overall risk level of this segment and what the average probability means in practice.
2. What the risk breakdown reveals — is risk concentrated or spread evenly across the group?
3. The likely business consequence if this group's risk is not addressed (e.g. attrition cost, loss of institutional knowledge).
4. 2-3 concrete, practical retention suggestions HR could act on for this specific segment.

Do not invent any numbers not provided above. Keep the response concise and avoid markdown formatting — give ans in a nice markdown format."""

In [301]:
from groq import Groq
client=Groq(api_key=os.getenv("GROQ_API_KEY"))
reponse=client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role":"user","content":prompt}]
)
from IPython.display import Markdown, display
display(Markdown(reponse.choices[0].message.content))

**Attrition risk snapshot**  
- **Overall chance of leaving:** **20 %** (about 1 in 5).  
- **Why this number shows up:** The model looks at a handful of key drivers and adds up their effects. For this employee the strongest signals are listed below, together with the actual values they have on record.

| Driver (how it moves the risk) | Employee’s value | How it pushes the risk |
|--------------------------------|------------------|------------------------|
| **Average monthly hours** (‑0.09) | **30 hours** per month | Working relatively few hours lowers the risk of leaving. |
| **Last performance evaluation** (‑0.07) | **0.7** (on a 0‑1 scale) | A solid recent rating also pulls the risk down. |
| **Number of projects** (‑0.04) | **6** projects | Managing a moderate workload reduces the chance of turnover. |
| **Satisfaction level** (‑0.03) | **0.8** (high) | The employee reports being quite satisfied, which cuts the risk. |
| **Time spent at the company** (+0.03) | **240 months** (20 years) | Very long tenure nudges the risk upward – long‑tenured staff sometimes look for new challenges or retirement options. |

**What the numbers mean in plain language**

- The employee is **highly satisfied (0.8)**, works **few hours (30 per month)**, and has **good recent performance (0.7)**. Those factors all make staying more likely.  
- The only thing that pushes the probability a little higher is the **very long tenure – 20 years with the firm**. After two decades, people often start thinking about career change, new responsibilities, or winding down.

**Take‑away for HR**

1. **Recognize the loyalty:** A 20‑year employee who is still satisfied and performing well is a valuable asset.  
2. **Offer growth or transition options:** Even though the risk is modest, consider a career‑development conversation – e.g., new leadership roles, mentorship programs, or a phased retirement plan – to keep the employee engaged.  
3. **Monitor workload:** The low average hours suggest a light current load; ensure the employee feels challenged enough without being overburdened.  

Overall, the model predicts a **low‑to‑moderate** attrition risk (≈20 %). By addressing the single upward driver—long tenure—and reinforcing the positive factors, you can help keep this seasoned employee on board.

### Building a Generalised Version for all variables

In [256]:
#Python has a built-in module called operator that turns comparison symbols into callable functions.
import operator
operator.lt(1, 5)   # same as 3 < 5, returns True
operator.gt(7, 5)   # same as 7 > 5, returns True

True

In [264]:
# Operations

ops= {"<": operator.lt,
    ">": operator.gt,
    "<=": operator.le,
    ">=": operator.ge,
    "==": operator.eq
}

test_filter=ops[">="](hr3["number_project"],6)
#print(test_filter)
filtered_group=hr3[test_filter]
print(len(filtered_group))

989


### wrapping the whole pipeline (filter → predict → tier → count) into one Python function

In [302]:
sign=("<", ">","<=",">=","==")


# Input Functions
variable_name= input(f"Enter variable name, Option:{model3.feature_names_in_}:")
select_operator= input(f"Enter Operator, Options:{sign}:")
filter_by_value=float(input("Enter value to filter by:"))



def attrition_by_filter(column, symbol, value):
    mask=ops[symbol](hr4[column],value)
    filtered_grp= hr4[mask]
    probs=model3.predict_proba(filtered_grp)[:,1]
    avg_prob=probs.mean()
    high_risk= probs[probs>=0.7]
    medium_risk= probs[(probs>=0.3)&(probs<0.7)]
    low_risk= probs[probs<0.3]
    return(len(filtered_grp),avg_prob,len(high_risk),len(medium_risk),len(low_risk))
    
result= attrition_by_filter(variable_name,select_operator,filter_by_value)

filtered_grp,avg,high,medium,low=result

#print(result)

prompt = f"""You are analyzing a segment of employees for an HR audience. Here is the data:
Filter applied: employees where {variable_name} {select_operator} {filter_by_value}
Total employees in this group: {filtered_grp}
Average predicted attrition probability for this group: {avg*100}%
Risk breakdown:

* High risk (>70% probability of leaving): {high} employees
* Medium risk (30-70% probability of leaving): {medium} employees
* Low risk (<30% probability of leaving): {low} employees

Using only the numbers above, write a plain-language summary for HR covering:

1. The overall risk level of this segment and what the average probability means in practice.
2. What the risk breakdown reveals — is risk concentrated or spread evenly across the group?
3. The likely business consequence if this group's risk is not addressed (e.g. attrition cost, loss of institutional knowledge).
4. 2-3 concrete, practical retention suggestions HR could act on for this specific segment.

Do not invent any numbers not provided above. Keep the response concise and formatted as clean markdown (headers, bullet points) for readability."""

from groq import Groq
client= Groq(api_key=os.getenv("GROQ_API_KEY"))
reponse=client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role":"user","content":prompt}]
)
from IPython.display import Markdown, display
display(Markdown(reponse.choices[0].message.content))


Enter variable name, Option:['satisfaction_level' 'last_evaluation' 'number_project'
 'average_montly_hours' 'time_spend_company' 'Work_accident'
 'promotion_last_5years' 'low' 'medium' 'RandD' 'accounting' 'hr'
 'management' 'marketing' 'product_mng' 'sales' 'support' 'technical']: hr
Enter Operator, Options:('<', '>', '<=', '>=', '=='): ==
Enter value to filter by: 0


## 1. Overall risk level  
- The 9,967 employees in this segment have an **average attrition probability of about 29.6 %**.  
- In practical terms, if the model’s forecast holds, roughly **1 in 3** of these workers is expected to leave within the next period.  

## 2. What the risk breakdown shows  
- **High‑risk employees (>70 % chance):** 2,202 individuals – the largest single group of “certain leavers.”  
- **Medium‑risk employees (30‑70 % chance):** 871 individuals – a smaller but still notable pool whose departure is plausible.  
- **Low‑risk employees (<30 % chance):** 6,894 individuals – the majority of the segment, but they still contribute to the overall average.  

Because the high‑risk count is more than double the medium‑risk count, the risk is **concentrated** in a relatively compact group rather than being evenly spread across all employees.

## 3. Likely business consequence if left unattended  
- Losing the 2,202 high‑risk workers (and a portion of the medium‑risk pool) would generate **significant turnover costs** – recruitment, onboarding, and training expenses – and would also erode **institutional knowledge** that is harder to replace.  
- Even the 871 medium‑risk employees represent potential gaps in critical roles, which could impact team continuity and productivity.

## 4. Practical retention actions for this segment  

| Action | Why it helps this group |
|--------|------------------------|
| **Targeted stay‑interview program** – schedule short, confidential conversations with the 2,202 high‑risk employees to uncover specific concerns (e.g., workload, career path, manager support). | Directly addresses the reasons behind the high probability of leaving and shows the organization values their input. |
| **Tailored development plans** – create individualized skill‑building or promotion roadmaps for the high‑ and medium‑risk employees, linking clear milestones to upcoming opportunities. | Increases engagement by giving a visible future within the company, reducing the appeal of external offers. |
| **Recognition and reward boost** – implement short‑term incentives (spot bonuses, extra PTO, public acknowledgment) for high‑risk employees who meet key performance targets. | Reinforces a sense of appreciation and can tip the balance for employees on the fence about staying. |

These steps focus resources on the **concentrated high‑risk group** while also supporting the medium‑risk cohort, helping to lower the segment’s overall attrition probability and protect the organization’s talent base.